In [2]:
import pandas as pd
import glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
#from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# 1. Load and concatenate all batches
files = glob.glob("data_batches/merged_*.pkl")
frames = []
for fp in files:
    with open(fp, "rb") as f:
        batch = pickle.load(f)

        for sample in batch:
            # assume sample = {"params": array([...]), "results": {"positivity_ok": [...], ...}}
            df_params = pd.DataFrame(sample["params"],                              
                                     columns=["m_phi","m_A","sin_ba","tan_beta","lambda6","lambda7","m12_2"])
            df_labels = pd.DataFrame(sample["results"])
            frames.append(pd.concat([df_params, df_labels], axis=1))
df = pd.concat(frames, ignore_index=True)




In [3]:
df

,m_phi,m_A,sin_ba,tan_beta,lambda6,lambda7,m12_2,positivity_ok,unitarity_ok,perturbativity_ok,...,w_total_h2,w_total_top,branching_ratio_h2_gaga,lambda1,lambda2,lambda3,lambda4,lambda5,lambda6,lambda7
0,228.845063,270.971833,0.963304,7184.869299,0.002274,-0.070648,4516.413347,0.0,0.0,0.0,...,1.427591e+06,1.338069,6.423276e-01,-4.069073e+10,0.301362,846.433770,787.847749,787.847749,0.002274,-0.070648
1,411.221860,430.160665,0.951660,4556.695240,-0.060391,0.059153,2442.553794,0.0,0.0,0.0,...,3.966933e+09,1.338069,3.015487e-07,-9.606350e+08,0.496223,3057.680690,45.764291,45.764291,-0.060391,0.059153
2,303.814782,135.311221,0.965834,8992.605290,-0.018244,-0.005245,4903.157389,0.0,0.0,0.0,...,1.165998e+14,1.338069,4.051009e-06,-6.060534e+10,0.342605,2142.753036,750.580737,750.580737,-0.018244,-0.005245
3,226.115625,262.590048,0.997185,1004.043304,-0.068794,-0.031611,4763.046797,0.0,0.0,0.0,...,4.659670e+02,1.338069,6.451989e-01,-9.467406e+07,0.260909,-17.355823,93.615992,93.615992,-0.068794,-0.031611
4,489.597056,379.196113,0.987537,7817.069250,-0.042552,0.085884,5590.702275,0.0,0.0,0.0,...,5.110712e+10,1.338069,1.859281e-05,-2.330195e+10,0.349113,3435.371153,382.823618,382.823618,-0.042552,0.085884
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690395,160.970855,483.149458,0.956804,2652.638894,-0.001726,0.007157,0.392232,1.0,0.0,0.0,...,7.281374e-01,1.338069,7.680721e-01,6.957694e+07,0.272036,123.255689,-13.325383,-13.325383,-0.001726,0.007157
690396,263.340012,351.313602,0.956379,7432.150363,-0.000997,0.004775,1.564501,1.0,0.0,0.0,...,7.812573e+08,1.338069,3.586852e-07,1.028612e+09,0.333290,1825.451076,-19.589340,-19.589340,-0.000997,0.004775
690397,382.133448,270.285811,0.967875,2446.189594,-0.002766,0.009979,0.432558,1.0,0.0,0.0,...,4.168757e+07,1.338069,6.650123e-06,8.653553e+07,0.393280,1268.758424,-13.393417,-13.393417,-0.002766,0.009979
690398,364.609113,346.275912,0.997232,3607.097642,0.001803,0.000958,1.691111,1.0,0.0,0.0,...,2.737555e+04,1.338069,3.661354e-04,4.956324e+07,0.268351,517.746256,-3.604932,-3.604932,0.001803,0.000958


## Constrains con $\lambda_j$

En caso de resolver el potencial completo en su forma generica, es posible obtener rapidamente constrains, reglas que deben cumplirse para evitar perder positividad, unitariedad y perturbatividad.

### Positividad:
$$
\lambda_1 > 0
$$

$$
\lambda_2 > 0
$$

$$
\lambda_3 > - \sqrt{\lambda_1 \lambda_2}
$$

$$
\lambda_3 + \lambda_4 - \lambda_5 > - \sqrt{\lambda_1 \lambda_2}
$$

In [4]:
print("## Missing values per column:")
print(df.isnull().sum(), "\n")

## Missing values per column:
m_phi                          0
m_A                            0
sin_ba                         0
tan_beta                       0
lambda6                        0
lambda7                        0
m12_2                          0
positivity_ok              14981
unitarity_ok               14981
perturbativity_ok          14981
w_h2_bb                    14981
w_h2_tautau                14981
w_h2_uu                    14981
w_h2_du                    14981
w_h2_ln                    14981
w_h2_vv                        0
w_h2_gaga                  14981
w_h2_Zga                   14981
w_h2_gg                    14981
w_h2_hh                    14981
w_total_h2                 14981
w_total_top                14981
branching_ratio_h2_gaga    14981
lambda1                    14981
lambda2                    14981
lambda3                    14981
lambda4                    14981
lambda5                    14981
lambda6                    14981
lambda7      

In [ ]:
df.dropna(axis=0, inplace=True)


In [6]:
# 2. Exploratory Data Analysis

print("## Label distribution:")
print(df[["positivity_ok","perturbativity_ok","unitarity_ok"]].mean(), "\n")

print(df[["positivity_ok","perturbativity_ok","unitarity_ok"]].sum(), "\n")

## Label distribution:
positivity_ok        0.445621
perturbativity_ok    0.000007
unitarity_ok         0.000037
dtype: float64 

positivity_ok        300981.0
perturbativity_ok         5.0
unitarity_ok             25.0
dtype: float64 



In [ ]:

print("## Basic stats for features:")
print(df[["m_phi","m_A","sin_ba","tan_beta","lambda6","lambda7","m12_2"]].describe(), "\n")


## Basic stats for features:
               m_phi            m_A         sin_ba       tan_beta  \
count  117396.000000  117396.000000  117396.000000  117396.000000   
mean      314.334745     314.985586       0.974996    5006.537637   
std       107.893349     106.790616       0.014434    2883.911465   
min       130.001736     130.004881       0.950000      10.071779   
25%       220.490361     222.514189       0.962500    2510.780852   
50%       310.994780     314.939108       0.974992    5007.539184   
75%       409.513357     407.471695       0.987494    7504.167481   
max       499.999412     499.999452       0.999999    9999.875040   

             lambda6        lambda6        lambda7        lambda7  \
count  117396.000000  117396.000000  117396.000000  117396.000000   
mean        0.000008       0.000008       0.000022       0.000022   
std         0.057732       0.057732       0.057723       0.057723   
min        -0.099999      -0.099999      -0.099999      -0.099999   
25% 

In [31]:

# 3. Train / test split
X = df[["m_phi","m_A","sin_ba","tan_beta","lambda6","lambda7","m12_2"]]
y = df[["positivity_ok","perturbativity_ok","unitarity_ok"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42,
                                                    stratify=y)

# 4. Define and train three multi-output classifiers
models = {
    "Random Forest": MultiOutputClassifier(
        RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    ),
    "XGBoost": MultiOutputClassifier(
        XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    ),
    "MLP": MultiOutputClassifier(
        MLPClassifier(hidden_layer_sizes=(64,64), max_iter=200, random_state=42)
    )
}

for name, clf in models.items():
    print(f"### Training {name}")
    clf.fit(X_train, y_train)
    print(f"### Evaluating {name}")
    y_pred = clf.predict(X_test)
    print(classification_report(y_test, y_pred, zero_division=0), "\n")


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.